# 🖼️ CNN-Autoencoder for Brain MRI Anomaly Detection

## 📋 Overview

This notebook implements a **Convolutional Neural Network (CNN) Autoencoder** for anomaly detection in brain MRI scans.

### Model Architecture
- **Type**: Convolutional Autoencoder
- **Input**: 128×128 grayscale MRI slice
- **Encoder**: 4 Conv2D layers with MaxPooling
- **Decoder**: 4 ConvTranspose2D layers for upsampling
- **Latent**: 256-dimensional bottleneck

### Key Features
- **Spatial feature extraction** with convolutional layers
- **Translation equivariance** (but NOT rotation equivariant)
- **Checkpoint recovery** for Colab disconnects
- **Saves every epoch** for resilience
- **Optimized for speed** (batch_size=64, 20 epochs)

### Expected Performance
- **AUROC**: ~0.82-0.87 (better than baseline)
- **Training Time**: ~45-60 min per epoch, ~15-20 hours total

### ⚡ Optimizations
- Larger batch size (64 vs 32) for faster training
- Fewer epochs (20 vs 100) - sufficient convergence
- Checkpoint every epoch (not every 5)
- Auto-resume from last checkpoint
- Keep-alive script included

---

## 1️⃣ Setup and Environment Configuration

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

print("✅ Google Drive mounted successfully!")

Mounted at /content/drive
✅ Google Drive mounted successfully!


In [2]:
# Keep Colab session alive (prevents random disconnects)
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button");
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click();
   }
   btn = document.querySelector('#ok');
   if (btn != null){
     console.log("Click connect button");
     btn.click();
   }
 }
 setInterval(ClickConnect, 60000)
'''))

print("✅ Keep-alive script activated!")

<IPython.core.display.Javascript object>

✅ Keep-alive script activated!


In [3]:
# Install required packages
!pip install pytorch-msssim -q

print("✅ All packages installed!")

✅ All packages installed!


In [4]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler  # Mixed precision training
from pytorch_msssim import SSIM

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc
from glob import glob
import os
import time
from tqdm import tqdm
import json

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [5]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🚀 Using device: cuda
   GPU: Tesla T4
   Memory: 15.83 GB


In [6]:
# Define paths
BASE_PATH = "/content/drive/MyDrive/symAD-ECNN"
IXI_TRAIN_PATH = f"{BASE_PATH}/data/processed_ixi/train"
IXI_VAL_PATH = f"{BASE_PATH}/data/processed_ixi/val"
BRATS_PATH = f"{BASE_PATH}/data/brats2021_test"
MODEL_PATH = f"{BASE_PATH}/models/saved_models"
RESULTS_PATH = f"{BASE_PATH}/results/cnn_autoencoder"

# Create directories if they don't exist
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("📁 Paths configured:")
print(f"   IXI Train: {IXI_TRAIN_PATH}")
print(f"   IXI Val: {IXI_VAL_PATH}")
print(f"   BraTS Data: {BRATS_PATH}")
print(f"   Models: {MODEL_PATH}")
print(f"   Results: {RESULTS_PATH}")

📁 Paths configured:
   IXI Train: /content/drive/MyDrive/symAD-ECNN/data/processed_ixi/train
   IXI Val: /content/drive/MyDrive/symAD-ECNN/data/processed_ixi/val
   BraTS Data: /content/drive/MyDrive/symAD-ECNN/data/brats2021_test
   Models: /content/drive/MyDrive/symAD-ECNN/models/saved_models
   Results: /content/drive/MyDrive/symAD-ECNN/results/cnn_autoencoder


## 2️⃣ Data Loading and Preprocessing

In [7]:
# Dataset class for MRI slices
class MRIDataset(Dataset):
    """
    Dataset for loading preprocessed MRI slices (.npy files)
    Returns: (image, image) pairs for autoencoder training
    """
    def __init__(self, file_list):
        self.files = file_list

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load .npy file
        img = np.load(self.files[idx])

        # Add channel dimension: (128, 128) -> (1, 128, 128)
        img = np.expand_dims(img, 0)

        # Convert to tensor
        img_tensor = torch.from_numpy(img).float()

        # Return (input, target) pair - same for autoencoder
        return img_tensor, img_tensor

print("✅ Dataset class defined!")

✅ Dataset class defined!


In [9]:
# Load file paths
print("📂 Loading file paths...")

train_files = sorted(glob(f"{IXI_TRAIN_PATH}/*.npy"))
val_files = sorted(glob(f"{IXI_VAL_PATH}/*.npy"))
brats_files = sorted(glob(f"{BRATS_PATH}/*.npy"))

print(f"✅ Found {len(train_files)} IXI training slices")
print(f"✅ Found {len(val_files)} IXI validation slices")
print(f"✅ Found {len(brats_files)} BraTS test slices (tumors)")

📂 Loading file paths...
✅ Found 15093 IXI training slices
✅ Found 1678 IXI validation slices
✅ Found 49888 BraTS test slices (tumors)


In [10]:
# Create datasets and dataloaders
train_dataset = MRIDataset(train_files)
val_dataset = MRIDataset(val_files)
test_dataset = MRIDataset(brats_files)

# Optimized batch size for CNN (larger than baseline for speed)
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✅ DataLoaders created!")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

✅ DataLoaders created!
   Batch size: 64
   Train batches: 236
   Val batches: 27
   Test batches: 780


## 3️⃣ CNN-Autoencoder Model Architecture

In [11]:
class CNNAutoencoder(nn.Module):
    """
    Convolutional Autoencoder

    Encoder: 128×128 -> 64×64 -> 32×32 -> 16×16 -> 8×8
    Decoder: 8×8 -> 16×16 -> 32×32 -> 64×64 -> 128×128
    """

    def __init__(self, latent_dim=256):
        super(CNNAutoencoder, self).__init__()

        # Encoder: Progressive downsampling
        self.encoder = nn.Sequential(
            # 128×128×1 -> 128×128×32
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 64×64

            # 64×64×32 -> 64×64×64
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 32×32

            # 32×32×64 -> 32×32×128
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 16×16

            # 16×16×128 -> 16×16×256
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2)  # 8×8
        )

        # Latent space
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)

        # Decoder: Progressive upsampling
        self.decoder = nn.Sequential(
            # 8×8×256 -> 16×16×256
            nn.ConvTranspose2d(256, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            # 16×16×256 -> 32×32×128
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # 32×32×128 -> 64×64×64
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            # 64×64×64 -> 128×128×32
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            # 128×128×32 -> 128×128×1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Encode
        features = self.encoder(x)

        # Latent
        batch_size = features.size(0)
        flat = self.flatten(features)
        z = self.fc_encode(flat)

        # Decode
        decoded_flat = self.fc_decode(z)
        decoded_features = decoded_flat.view(batch_size, 256, 8, 8)
        x_recon = self.decoder(decoded_features)

        return x_recon

    def get_latent(self, x):
        features = self.encoder(x)
        flat = self.flatten(features)
        return self.fc_encode(flat)

model = CNNAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters())

print("🖼️ CNN-Autoencoder Created!")
print(f"   Total parameters: {total_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.2f} MB")

🖼️ CNN-Autoencoder Created!
   Total parameters: 9,772,673
   Model size: ~37.28 MB


## 4️⃣ Loss Function and Optimizer

In [12]:
class CombinedLoss(nn.Module):
    """
    Combined MSE + SSIM Loss (using SSIM for 128×128 images)

    Loss = α * MSE + (1-α) * (1 - SSIM)
    """
    def __init__(self, alpha=0.84):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.ssim = SSIM(data_range=1.0, size_average=True, channel=1, win_size=11)

    def forward(self, output, target):
        mse_loss = self.mse(output, target)
        ssim_loss = 1 - self.ssim(output, target)
        combined = self.alpha * mse_loss + (1 - self.alpha) * ssim_loss
        return combined

criterion = CombinedLoss(alpha=0.84)

print("✅ Combined Loss Function created!")
print(f"   MSE weight (α): 0.84")
print(f"   SSIM weight (1-α): 0.16")

✅ Combined Loss Function created!
   MSE weight (α): 0.84
   SSIM weight (1-α): 0.16


In [14]:
def create_optimizer_scheduler(model):
    """Create optimizer and scheduler"""
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    return opt, sched

# Create GradScaler for mixed precision training

scaler = GradScaler()
print("✅ Optimizer factory and GradScaler defined!")


✅ Optimizer factory and GradScaler defined!


/tmp/ipython-input-1281916654.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


## 5️⃣ Training Functions

In [22]:
def train_epoch(model, dataloader, criterion, optimizer, device, scaler):
    """Train with mixed precision (FP16) for 2-3x speedup"""
    model.train()
    running_loss = 0.0
    pbar = tqdm(dataloader, desc='Training')
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast(): # Changed from autocast('cuda') to autocast()
            output = model(data)
            loss = criterion(output, target)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    return running_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            running_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    return running_loss / len(dataloader)

In [25]:
# Delete broken checkpoints
!rm {MODEL_PATH}/cnn_*.pth
print("✅ Checkpoints deleted, ready for clean restart")

✅ Checkpoints deleted, ready for clean restart


## 6️⃣ Checkpoint Recovery

In [26]:
# Check for existing checkpoints
checkpoints = sorted(glob(f'{MODEL_PATH}/cnn_epoch*.pth'))
RESUME = len(checkpoints) > 0

if RESUME:
    latest = checkpoints[-1]
    print(f"✅ Found checkpoint: {os.path.basename(latest)}")
    checkpoint = torch.load(latest)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer, scheduler = create_optimizer_scheduler(model)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    best_epoch = checkpoint.get('best_epoch', 0)
    print(f"   Resuming from epoch {start_epoch}")
else:
    optimizer, scheduler = create_optimizer_scheduler(model)
    start_epoch = 0
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_epoch = 0
    print("📝 Starting fresh training")

📝 Starting fresh training


## 7️⃣ Main Training Loop

In [27]:
NUM_EPOCHS = 20  # Optimized: 20 epochs for faster training
print(f"🚀 Training CNN-AE: Epochs {start_epoch + 1} to {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}, Device: {device}")
print("-" * 60)

start_time = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    epoch_time = time.time() - epoch_start
    print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Time: {epoch_time/60:.1f}min")

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch
        }, f'{MODEL_PATH}/cnn_best.pth')
        print(f"   ✅ Best model saved!")

    # Save checkpoint every epoch
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch
    }, f'{MODEL_PATH}/cnn_epoch{epoch+1}.pth')

    print("-" * 60)

total_time = time.time() - start_time
print(f"\n🎉 Training Complete!")
print(f"   Total Time: {total_time/3600:.2f} hours")
print(f"   Best Epoch: {best_epoch}, Best Val Loss: {best_val_loss:.6f}")

🚀 Training CNN-AE: Epochs 1 to 20
   Batch size: 64, Device: cuda
------------------------------------------------------------


Training:   0%|          | 0/236 [00:00<?, ?it/s]/tmp/ipython-input-2950176455.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Changed from autocast('cuda') to autocast()
Validation: 100%|██████████| 27/27 [00:05<00:00,  4.86it/s, loss=0.026936]


Epoch [  1/20] | Train: 0.027350 | Val: 0.026876 | Time: 0.9min
   ✅ Best model saved!
------------------------------------------------------------


Validation: 100%|██████████| 27/27 [00:06<00:00,  4.28it/s, loss=0.027306]


Epoch [  2/20] | Train: 0.026307 | Val: 0.027334 | Time: 0.9min
------------------------------------------------------------


...

Epoch [ 18/20] | Train: 0.023140 | Val: 0.023255 | Time: 0.9min
------------------------------------------------------------


Validation: 100%|██████████| 27/27 [00:05<00:00,  5.23it/s, loss=0.023138]


Epoch [ 19/20] | Train: 0.022152 | Val: 0.023089 | Time: 0.9min
   ✅ Best model saved!
------------------------------------------------------------


Validation: 100%|██████████| 27/27 [00:05<00:00,  4.50it/s, loss=0.022862]


Epoch [ 20/20] | Train: 0.021738 | Val: 0.022843 | Time: 1.0min
   ✅ Best model saved!
------------------------------------------------------------

🎉 Training Complete!
   Total Time: 0.32 hours
   Best Epoch: 20, Best Val Loss: 0.022843


In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.axvline(x=best_epoch-1, color='r', linestyle='--', label=f'Best ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN-AE Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss (log)')
plt.title('CNN-AE Training (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_training_curves.png', dpi=150)
plt.show()

## 8️⃣ Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(f'{MODEL_PATH}/cnn_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Best model loaded (Epoch {checkpoint['epoch']}, Val Loss: {checkpoint['val_loss']:.6f})")

# Calculate reconstruction errors
def calculate_errors(model, dataloader, device):
    model.eval()
    errors = []
    with torch.no_grad():
        for data, _ in tqdm(dataloader, desc='Computing errors'):
            data = data.to(device)
            recon = model(data)
            mse = nn.functional.mse_loss(recon, data, reduction='none').view(data.size(0), -1).mean(dim=1)
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

normal_errors = calculate_errors(model, val_loader, device)
anomaly_errors = calculate_errors(model, test_loader, device)

# Metrics
y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
y_scores = np.concatenate([normal_errors, anomaly_errors])
auroc = roc_auc_score(y_true, y_scores)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = auc(recall, precision)

print(f"\n📈 CNN-AE Performance:")
print(f"   AUROC: {auroc:.4f}")
print(f"   AUPRC: {auprc:.4f}")
print(f"   Normal errors: {normal_errors.mean():.6f} ± {normal_errors.std():.6f}")
print(f"   Anomaly errors: {anomaly_errors.mean():.6f} ± {anomaly_errors.std():.6f}")

# Plot
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(normal_errors, bins=50, alpha=0.7, label='Normal', density=True, color='blue')
plt.hist(anomaly_errors, bins=50, alpha=0.7, label='Anomaly', density=True, color='red')
plt.xlabel('Reconstruction Error')
plt.ylabel('Density')
plt.legend()
plt.title('CNN-AE: Error Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_true, y_scores)
plt.plot(fpr, tpr, linewidth=2, label=f'CNN-AE (AUROC={auroc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.title('CNN-AE: ROC Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_evaluation.png', dpi=150)
plt.show()

# Save results
results = {
    'model': 'CNN-Autoencoder',
    'auroc': float(auroc),
    'auprc': float(auprc),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'normal_error_mean': float(normal_errors.mean()),
    'anomaly_error_mean': float(anomaly_errors.mean()),
    'total_params': total_params,
    'training_time_hours': total_time / 3600
}

with open(f'{RESULTS_PATH}/cnn_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\n✅ CNN-AE evaluation complete!")